# Laboratorio 05 — Dependencias temporales entre cursos

## Objetivo

En este laboratorio estudiaremos la estructura temporal de las matrículas **antes de escoger un modelo**.

La hipótesis central es que la matrícula de un curso en un semestre puede depender de:

- su propia matrícula en semestres anteriores;
- la estacionalidad entre primer y segundo semestre;
- la matrícula previa de otros cursos;
- movimientos conjuntos asociados a cohortes, prerrequisitos u otras relaciones académicas.

El objetivo no es todavía entrenar el modelo final, sino comprender qué información histórica puede ser predictiva y qué características convendría construir posteriormente.

> **Importante:** correlación no implica causalidad. Además, una correlación entre dos cursos en el mismo período no es necesariamente utilizable para predecir, porque al momento de hacer la predicción todavía no conocemos las matrículas del período futuro.


## 1. Carga del dataset preparado

Partimos del mismo archivo raw del laboratorio anterior, pero reutilizamos las funciones del proyecto para cargar y preparar los períodos académicos.

El notebook se mantiene como espacio de **exploración**. Las transformaciones que ya fueron formalizadas permanecen en `src/matriculas/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matriculas.data import load_raw_data
from matriculas.prepare import prepare_period
from matriculas.validate import validate_dataset

RAW_DIR = Path("../data/raw")
DATASET = RAW_DIR / "reporte1_31_08_26" / "reporte1_31_08_26.xlsx"

df = load_raw_data(DATASET)
df = prepare_period(df)
validate_dataset(df)

df.head()

## 2. Revisar la cobertura temporal

Antes de estudiar dependencias conviene verificar cuántos cursos aparecen en cada período. Un período incompleto puede distorsionar correlaciones y análisis posteriores.

En particular, observaremos el número de cursos y la matrícula total por semestre.


In [ ]:
period_summary = (
    df.groupby(["time_index", "period"], as_index=False)
    .agg(
        courses=("course_code", "nunique"),
        total_enrollment=("enrollment", "sum"),
    )
    .sort_values("time_index")
)

period_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(period_summary["time_index"], period_summary["courses"], marker="o")
ax.set_xlabel("Índice temporal")
ax.set_ylabel("Número de cursos")
ax.set_title("Cursos observados por período")
ax.grid(alpha=0.25)
plt.show()

### Decisión sobre períodos incompletos

No eliminaremos períodos automáticamente. Primero los identificamos.

Para el análisis de dependencias conviene trabajar únicamente con períodos que representen una fotografía suficientemente completa del semestre. El siguiente bloque permite excluir explícitamente períodos que sepamos que están incompletos.

En el archivo actual aparece `202710` con una cobertura muy inferior a los semestres anteriores, por lo que se excluye **solo de este análisis exploratorio**. Esta decisión deberá formalizarse posteriormente como una regla de calidad/completitud si se confirma su significado.


In [ ]:
EXCLUDED_PERIODS = {202710}

analysis_df = df.loc[~df["period"].isin(EXCLUDED_PERIODS)].copy()

analysis_df["period"].sort_values().unique()

## 3. Construcción de la matriz período × curso

Para estudiar relaciones entre cursos transformaremos los datos desde formato largo:

| period | course_code | enrollment |
|---|---|---:|
| 202510 | A | 100 |
| 202510 | B | 40 |
| 202520 | A | 90 |

a una matriz:

| period | A | B | ... |
|---|---:|---:|---:|
| 202510 | 100 | 40 | ... |
| 202520 | 90 | 45 | ... |

Cada columna será la serie temporal de un curso.

**No rellenamos todavía los valores faltantes con cero.** La ausencia de un curso en un período no necesariamente significa matrícula cero; podría significar que no se ofreció o que no está presente en el reporte. Para correlaciones, pandas puede trabajar con observaciones coincidentes.


In [ ]:
course_matrix = (
    analysis_df.pivot(
        index="time_index",
        columns="course_code",
        values="enrollment",
    )
    .sort_index()
)

course_matrix.shape

In [ ]:
course_matrix.iloc[:5, :8]

## 4. ¿Cuánto histórico tiene cada curso?

No todos los cursos aparecen durante toda la historia. Una correlación calculada con dos o tres observaciones es poco confiable.

Contaremos en cuántos períodos aparece cada curso y trabajaremos inicialmente con cursos que tengan una cantidad mínima de observaciones.


In [ ]:
course_history = (
    course_matrix.notna()
    .sum()
    .sort_values(ascending=False)
    .rename("periods_observed")
)

course_history.describe()

In [ ]:
MIN_PERIODS = 8

eligible_courses = course_history[course_history >= MIN_PERIODS].index

matrix = course_matrix[eligible_courses]

print(f"Cursos totales: {course_matrix.shape[1]}")
print(f"Cursos con al menos {MIN_PERIODS} períodos: {matrix.shape[1]}")

## 5. Autodependencia: ¿un curso depende de su propio pasado?

Comenzamos por la relación más sencilla:

\[
y_{i,t} \quad 	ext{vs.} \quad y_{i,t-k}
\]

donde `k=1` representa el semestre inmediatamente anterior y `k=2` el mismo semestre del año anterior.

Esto permitirá comparar la importancia potencial de `lag_1` y `lag_2`.


In [ ]:
def autocorrelation_by_course(data: pd.DataFrame, lag: int) -> pd.Series:
    correlations = {}

    for course in data.columns:
        series = data[course]
        correlations[course] = series.corr(series.shift(lag))

    return pd.Series(correlations, name=f"autocorr_lag_{lag}")


autocorr_1 = autocorrelation_by_course(matrix, lag=1)
autocorr_2 = autocorrelation_by_course(matrix, lag=2)

autocorr = pd.concat([autocorr_1, autocorr_2], axis=1)
autocorr.describe()

In [ ]:
autocorr.dropna().sort_values("autocorr_lag_2", ascending=False).head(20)

### Interpretación

Si `lag_2` resulta sistemáticamente más informativo que `lag_1`, sería consistente con una estacionalidad anual: primer semestre con primer semestre y segundo semestre con segundo semestre.

Esto todavía no demuestra que `lag_2` produzca mejores predicciones. Esa comparación se hará posteriormente mediante backtesting.


## 6. Correlación contemporánea entre cursos

Ahora estudiaremos si dos cursos tienden a aumentar o disminuir conjuntamente **en el mismo período**:

\[
corr(y_{i,t}, y_{j,t})
\]

Este análisis puede revelar grupos de cursos asociados a cohortes o estructuras académicas comunes.

Sin embargo, estas correlaciones **no pueden utilizarse directamente como features para predecir el mismo período**, porque \(y_{j,t}\) tampoco se conoce todavía.


In [ ]:
# Exigimos varias observaciones coincidentes para evitar correlaciones
# calculadas con muy pocos períodos.
MIN_COMMON_PERIODS = 8

corr_same_period = matrix.corr(min_periods=MIN_COMMON_PERIODS)

corr_same_period.shape

Para evitar imprimir una matriz de miles por miles, seleccionaremos un curso y buscaremos los cursos con los que presenta mayor correlación contemporánea.

Puedes cambiar `TARGET_COURSE` por cualquier código existente.


In [ ]:
TARGET_COURSE = matrix.columns[0]

same_period_relations = (
    corr_same_period[TARGET_COURSE]
    .drop(labels=TARGET_COURSE)
    .dropna()
    .sort_values(ascending=False)
)

same_period_relations.head(15)

In [ ]:
top_courses = [TARGET_COURSE, *same_period_relations.head(4).index]

fig, ax = plt.subplots(figsize=(12, 5))

for course in top_courses:
    ax.plot(matrix.index, matrix[course], marker="o", label=course)

ax.set_xlabel("Índice temporal")
ax.set_ylabel("Estudiantes")
ax.set_title(f"Cursos relacionados contemporáneamente con {TARGET_COURSE}")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 7. Dependencia cruzada temporal

Para predicción es más interesante estudiar:

\[
corr(y_{i,t}, y_{j,t-1})
\]

porque al predecir el período `t` sí conocemos la matrícula del curso `j` en `t-1`.

También podemos estudiar:

\[
corr(y_{i,t}, y_{j,t-2})
\]

para capturar relaciones anuales o secuencias académicas más largas.

Construiremos una función que, dado un curso objetivo, busque qué cursos del pasado presentan mayor correlación con su matrícula futura.


In [ ]:
def lagged_course_correlations(
    data: pd.DataFrame,
    target_course: str,
    lag: int = 1,
    min_observations: int = 8,
) -> pd.DataFrame:
    target = data[target_course]
    rows = []

    for source_course in data.columns:
        source_lagged = data[source_course].shift(lag)

        pair = pd.concat(
            [target.rename("target"), source_lagged.rename("source")],
            axis=1,
        ).dropna()

        if len(pair) < min_observations:
            continue

        rows.append(
            {
                "source_course": source_course,
                "lag": lag,
                "correlation": pair["target"].corr(pair["source"]),
                "observations": len(pair),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("correlation", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
lag1_relations = lagged_course_correlations(
    matrix,
    target_course=TARGET_COURSE,
    lag=1,
    min_observations=MIN_COMMON_PERIODS,
)

lag1_relations.head(15)

In [ ]:
lag2_relations = lagged_course_correlations(
    matrix,
    target_course=TARGET_COURSE,
    lag=2,
    min_observations=MIN_COMMON_PERIODS,
)

lag2_relations.head(15)

## 8. Comparar autodependencia y dependencia de otros cursos

Para el curso objetivo podemos comparar:

- su propio `lag_1`;
- su propio `lag_2`;
- los cursos externos más relacionados con un semestre de retraso;
- los cursos externos más relacionados con dos semestres de retraso.

Esto permite comenzar a responder si el curso parece explicarse principalmente por su propia historia o si existe información relevante en otros cursos.


In [ ]:
own_relations = pd.DataFrame(
    {
        "relationship": ["own_lag_1", "own_lag_2"],
        "correlation": [
            matrix[TARGET_COURSE].corr(matrix[TARGET_COURSE].shift(1)),
            matrix[TARGET_COURSE].corr(matrix[TARGET_COURSE].shift(2)),
        ],
    }
)

own_relations

## 9. Cuidado con el *data leakage*

Hasta ahora hemos utilizado todo el histórico disponible porque estamos haciendo **análisis exploratorio**.

Pero cuando estas relaciones se conviertan en un mecanismo de selección de features, no podremos calcularlas usando el período de evaluación.

Por ejemplo, si queremos evaluar una predicción para `202620`, la selección de cursos relacionados deberá hacerse exclusivamente con períodos anteriores:

```text
TRAIN
201510 ───────────────── 202610
          ↓
calcular relaciones
seleccionar features

TEST
202620
```

No debemos usar `202620` para decidir qué cursos están correlacionados y luego utilizar ese mismo período para medir el desempeño.

En el siguiente laboratorio, esta separación temporal será parte explícita del pipeline de entrenamiento.


## 10. Una primera visión de las features

Este análisis sugiere que para predecir la matrícula de un curso \(i\) podríamos construir información como:

\[
X_{i,t} =
[
y_{i,t-1},
y_{i,t-2},
y_{j_1,t-1},
y_{j_2,t-1},
\dots,
semester_t
]
\]

donde \(j_1,j_2,\dots\) son otros cursos cuya historia aporta información predictiva.

Todavía no fijaremos el número de cursos relacionados ni el método definitivo de selección. Primero debemos comprobar mediante backtesting si esas relaciones mejoran realmente la predicción.


## 11. Ejercicio exploratorio

Selecciona varios cursos con suficiente historia y compara:

1. su serie temporal;
2. autocorrelación con `lag_1`;
3. autocorrelación con `lag_2`;
4. cursos con mayor correlación contemporánea;
5. cursos con mayor correlación cruzada en `lag_1`;
6. cursos con mayor correlación cruzada en `lag_2`.

Preguntas para discutir:

- ¿`lag_1` o `lag_2` parece más informativo?
- ¿Aparecen otros cursos más correlacionados con el futuro del curso objetivo que su propia historia?
- ¿Las relaciones encontradas tienen una explicación académica plausible?
- ¿Hay correlaciones altas sustentadas por muy pocas observaciones?
- ¿Qué relaciones serían realmente utilizables al momento de hacer una predicción?


## Conclusión

El problema de predicción de matrículas no debe tratarse necesariamente como miles de series temporales independientes.

Cada curso posee memoria propia, pero también puede compartir información con otros cursos debido a cohortes, secuencias curriculares, cursos comunes y otros fenómenos institucionales.

Este laboratorio permitió distinguir tres relaciones diferentes:

- **autodependencia temporal:** un curso respecto a su propio pasado;
- **correlación contemporánea:** cursos que se mueven conjuntamente;
- **dependencia cruzada temporal:** información de otros cursos anteriores potencialmente útil para predecir un curso futuro.

El siguiente paso será transformar estas observaciones en un **dataset supervisado reproducible**, separar entrenamiento y evaluación temporalmente y comparar baselines contra un primer modelo. Solo las relaciones calculadas a partir del conjunto de entrenamiento podrán participar en la selección de características.
